# 08 · Trabajo exploratorio

**Modalidad:** domiciliario, obligatorio.

---

## Qué se pide

Elegir un material, explorarlo con spaCy y **documentar el recorrido**.

No hay una respuesta correcta ni una lista de ejercicios que completar. Lo que
se evalúa es que hayas hecho una elección sobre un material y lo hayas explorado
conscientemente: que sepas qué probaste, qué salió, qué no salió y por qué
seguiste por donde seguiste.

Un hallazgo modesto con las decisiones a la vista vale más que una tabla vistosa
sin fundamentación.

---

## Condiciones de entrega

Cinco secciones. Todas tienen que estar.

### 1 · La elección

Qué material elegiste y **por qué ese y no otro**. Qué esperabas encontrar antes
de mirar.

Escribí esto **antes** de correr nada y no lo edites después. La distancia entre
lo que esperabas y lo que pasó es la mitad del trabajo.

**Esto es un ensayo del título-tesis.** En el integrador vas a tener que formular uno en
serio, con su variable medible y su refutación declarada (`07`, sección 7). Acá lo
hacés solo, sobre un material que no elegís en equipo y sin nota de por medio. Aprovechá
para equivocarte barato.

### 2 · La exploración

Qué operaciones de spaCy corriste sobre ese material y qué devolvieron.
Incluidas las que no devolvieron nada útil.

### 3 · Las perillas que moviste

Qué cambiaste y qué pasó. El modelo, el filtro de POS, la lista del
`PhraseMatcher`, el umbral, los componentes que apagaste, el tamaño de la
muestra, el nivel de agregación.

Cada perilla con su antes y su después.

### 4 · Lo que descartaste

**Al menos una línea que abriste y cerraste, con el motivo.**

Esta es la sección que más importa. Un callejón sin salida documentado vale tanto
como un hallazgo. Si no hay nada acá, no hubo exploración: hubo ejecución de un
pipeline.

### 5 · Dónde quedó

Qué pregunta te podés hacer ahora que antes no podías. Qué haría falta para
contestarla.

---

## Sobre usar IA para este trabajo

Podés usarla. Lo que no se puede es que reemplace las secciones 3 y 4, porque un
modelo de lenguaje no tiene la información que ahí se pide: **qué probaste vos,
qué falló en tu corpus, qué decidiste descartar.**

Si usaste IA, decilo y contá para qué. Eso también es parte de documentar el
recorrido.

## El menú de materiales

Elegir es el primer acto de criterio. Cada opción permite unas preguntas y
bloquea otras. Elegí sabiendo qué ganás y qué resignás.

### A · Casciari por párrafos

`datos/corpus.jsonl` — 5.428 párrafos, 2004 a 2015.

**Permite:** serie temporal de doce años, léxico rioplatense, todo lo que ya
construiste en la unidad.

**No permite:** nada sobre el texto como unidad. No hay títulos, no hay límites
de obra, no se puede comparar un cuento con otro. Está todo en `datos/FUENTE.md`.

### B · Cuentos latinoamericanos

`datos/cuentos_latinoamericanos.csv` — 719 cuentos de 57 autores, scrapeados de
Ciudad Seva por Karen Palacio.

**Permite:** comparar entre obras y entre autores. Cada fila es un cuento con
`title`, `author` y `link`.

**No permite:** serie temporal. No trae año de publicación.

**Atención:** este dataset lo armó otra persona y **tiene defectos**. Auditarlo
antes de usarlo es parte del trabajo.

### C · Noticias

Las armás vos con `trafilatura`, como en la primera clase de la materia.

**Permite:** fuentes, atribución, actores en disputa. Es el único material donde
preguntas como "¿quién habla y quién es hablado?" tienen sentido pleno.

**No permite:** empezar rápido. Hay que construir el corpus primero.

### D · Uno propio

Material de otra materia, de tu trabajo, de una investigación tuya.

**Permite:** que el trabajo te sirva para algo más que la nota.

**No permite:** que yo te anticipe con qué te vas a encontrar.

## Punto de partida

Las celdas que siguen cargan cada opción. Corré la que corresponda y seguí desde
ahí.

In [ ]:
# spacy: la librería de análisis lingüístico.
import spacy

# pandas: para trabajar los corpus como tablas.
import pandas as pd

# Counter: para contar.
from collections import Counter

# Nuestros componentes propios del cuaderno 05.
import componentes

# pipeline_propio: el pipeline configurado, con reglas y correcciones, que
# guardaste en el cuaderno 05.
nlp = spacy.load("pipeline_propio")

# Si preferís el modelo de fábrica, sin nuestras reglas:
# nlp = spacy.load("es_core_news_lg")

# pipe_names: la lista de componentes que tiene el pipeline cargado
print("pipeline:", nlp.pipe_names)

### Opción A — Casciari

Ya procesado. Cargar el `DocBin` tarda dos segundos.

In [ ]:
# DocBin: el contenedor de documentos ya procesados.
from spacy.tokens import DocBin

# lines=True: el archivo .jsonl tiene un registro JSON por renglón
corpus_casciari = pd.read_json("datos/corpus.jsonl", lines=True)

# Creamos un banco vacío y lo llenamos con lo que guardó el cuaderno 06.
banco = DocBin()
banco.from_disk("datos/corpus.spacy")

# get_docs() pide el vocab del pipeline para volver a armar cada Doc.
generador = banco.get_docs(nlp.vocab)
documentos_casciari = list(generador)

cantidad_parrafos = len(documentos_casciari)
# :, separa los miles con una coma
print("párrafos:", f"{cantidad_parrafos:,}")

# unique(): cada año una sola vez. sorted(): de menor a mayor
anios_distintos = corpus_casciari["anio"].unique()
anios_del_corpus = sorted(anios_distintos)
print("años    :", anios_del_corpus)

### Opción B — Cuentos latinoamericanos

In [ ]:
cuentos = pd.read_csv("datos/cuentos_latinoamericanos.csv")

cantidad_de_filas = len(cuentos)
# columns: los nombres de las columnas de la tabla
nombres_de_columnas = list(cuentos.columns)
# nunique(): cuántos valores distintos hay en la columna
cantidad_de_autores = cuentos["author"].nunique()

print("filas    :", cantidad_de_filas)
print("columnas :", nombres_de_columnas)
print("autores  :", cantidad_de_autores)
print()

# value_counts(): cuántos cuentos tiene cada autor, de mayor a menor
cuentos_por_autor = cuentos["author"].value_counts()
# head(12): los primeros 12
autores_con_mas_cuentos = cuentos_por_autor.head(12)
print(autores_con_mas_cuentos)

**Antes de usar este dataset, auditalo.** La celda siguiente es un punto de
partida, no una lista completa. Buscá vos qué más está mal.

In [ ]:
# ¿Cuántos títulos aparecen más de una vez?
titulos_vistos = []
titulos_repetidos = 0
for titulo in cuentos["title"]:
    if titulo in titulos_vistos:
        titulos_repetidos = titulos_repetidos + 1
    else:
        titulos_vistos.append(titulo)

print("títulos duplicados:", titulos_repetidos)

# ¿Cuántos textos son sospechosamente cortos?
textos_cortos = 0
for texto in cuentos["text"]:
    # str(): si la celda está vacía, pandas la lee como un número (NaN).
    # La pasamos a texto para poder medirla igual.
    texto_como_str = str(texto)
    if len(texto_como_str) < 200:
        textos_cortos = textos_cortos + 1

print("textos de menos de 200 caracteres:", textos_cortos)

# ¿Qué valores tiene la columna metadata?
print()
print("valores de la columna metadata:")
valores_de_metadata = cuentos["metadata"].value_counts()
print(valores_de_metadata)

Mirá los valores de `metadata`. La mayoría son etiquetas de género, pero hay
dos que claramente no lo son: una fecha de nacimiento y una nota entre
paréntesis. Son errores del scraping que quedaron en el dataset.

Y fijate en la primera columna, `Unnamed: 0`: es el índice de pandas que alguien
guardó sin querer al exportar el CSV.

Ninguno de estos problemas te impide trabajar. Pero si armás una tabla por género
usando `metadata` sin mirarla antes, vas a reportar categorías que no existen.

### Opción C — Noticias

Armás el corpus con lo de la primera clase. Guardalo en el mismo formato
`.jsonl` con procedencia, como hiciste allá.

In [ ]:
# trafilatura: para extraer el texto principal de una página.
# Si no lo tenés instalado:  uv pip install trafilatura
#
# import trafilatura
#
# URLS = [
#     "https://...",
#     "https://...",
# ]
#
# Escribí tu cargador acá, reusando el de la clase de scraping.
# Guardá cada nota con url, medio, título, fecha y texto.

## Un repertorio de operaciones

**Esto no es una checklist.** Son disparadores para que tengas de dónde arrancar.
Elegí dos o tres que sirvan a tu pregunta e ignorá el resto.

**Sobre las palabras**
- Frecuencias por lema, filtrando por categoría gramatical
- Comparar el ranking con distintos filtros de POS: solo `NOUN`, solo `VERB`, solo `ADJ`
- Buscar palabras que estén en un subcorpus y no en otro

**Sobre la morfología**
- Distribución de tiempos verbales
- Persona gramatical: ¿quién narra?
- Género y número de los sustantivos

**Sobre la sintaxis**
- Sujetos de verbos de acción: ¿quién actúa?
- Adjetivos que modifican a un sustantivo determinado
- Verbos de reporte y sus sujetos: ¿quién tiene voz?

**Sobre las entidades**
- Personas, lugares y organizaciones más frecuentes
- Coapariciones: qué entidades aparecen juntas
- Reglas propias para lo que el modelo no conoce

**Sobre los vectores**
- Palabras más parecidas a una que te interese
- Documentos más parecidos entre sí

**Sobre la herramienta misma**
- Correr lo mismo con `sm` y con `lg` y comparar
- Medir precisión sobre una muestra anotada a mano
- Ver qué cambia al mover el umbral, la muestra o el nivel de agregación

## Dos maneras de leer la misma salida

Varias de esas operaciones sirven a dos oficios distintos. Vale la pena saber en
cuál estás parado, porque cambia qué tenés que entregar.

| | investigador social | cientista de datos |
|---|---|---|
| **qué busca** | operacionalizar una pregunta conceptual de forma defendible | convertir texto en tablas y features, a escala y de forma reproducible |
| **con qué** | `dep_` para agencia · verbos de reporte para atribución · `morph` para impersonalidad · modificadores para nominación | `nlp.pipe` · `disable` · `DocBin` · reglas + NER · lematizar antes de vectorizar |
| **qué entrega** | una interpretación con la evidencia y las decisiones a la vista | una tabla, un pipeline versionado y una métrica |
| **qué comparten** | que el modelo no es neutro y hay que auditarlo antes de creerle | |

Los dos pueden hacer el mismo trabajo práctico. Lo que cambia es dónde ponen el
peso.

## Qué se ve como exploración consciente y qué no

Dos ejemplos cortos sobre el mismo material, para que se note la diferencia.

---

**Ejemplo 1**

> *Conté los sustantivos más frecuentes del corpus de Casciari con spaCy,
> filtrando stopwords. Los diez más frecuentes son: año, cosa, casa, vida, mundo,
> noche, amigo, nombre, mujer, tiempo. Se observa que predominan términos de la
> vida cotidiana, lo que confirma el carácter costumbrista de la obra.*

Corrió una celda y escribió una conclusión. No sabemos sobre cuántos párrafos,
con qué modelo, qué pasó si cambiaba el filtro, ni por qué "cosa" —que no dice
nada— quedó segundo.

---

**Ejemplo 2**

> *Quería ver si los temas cambiaban entre 2004 y 2015. Primero conté sustantivos
> por año sobre 200 párrafos de cada uno, y los rankings salieron casi idénticos:
> "cosa", "año", "casa" arriba de todo en los doce. Sospeché que el problema era
> que "cosa" y "año" son palabras de relleno que aparecen en cualquier texto.*
>
> *Probé sacarlas a mano y el ranking se movió poco. Después probé filtrar por
> `PROPN` en lugar de `NOUN` —nombres propios en vez de sustantivos comunes— y
> ahí sí apareció algo: en 2004 y 2005 dominan nombres de personajes de ficción
> (Mirta, Bertotti, Lelé) y a partir de 2010 aparecen "Orsai" y "revista".*
>
> *Descarté seguir por la línea de los sustantivos comunes: con este corpus y esta
> cantidad de párrafos no discrimina entre años, y para saber si es un problema
> de la muestra o del método haría falta procesar los años completos, que son
> cuarenta minutos de cómputo que no tenía.*
>
> *Con lo que aprendí ahora podría preguntar cuándo aparece por primera vez cada
> nombre propio, que es una pregunta sobre la biografía del proyecto y no sobre
> el vocabulario.*

Este segundo texto tiene lo que se pide: una expectativa previa, un intento que
falló, un diagnóstico, un segundo intento, una línea descartada **con el motivo**,
y una pregunta nueva.

Fijate que el hallazgo es modesto. No importa.

## Formato de entrega

Un cuaderno `.ipynb` con las cinco secciones en celdas de markdown, y el código
que las sostiene entre medio.

**Corré el cuaderno entero de arriba a abajo antes de entregar.** Si una celda
falla, tiene que fallar a propósito y estar explicado.

Nombre del archivo: `apellido_exploracion.ipynb`.

## Tu trabajo empieza acá

## 1 · La elección

*(Escribí acá qué material elegiste, por qué, y qué esperás encontrar. Antes de
correr nada.)*

In [ ]:
# TU CÓDIGO ACÁ

## 2 · La exploración

In [ ]:
# TU CÓDIGO ACÁ

## 3 · Las perillas que moví

In [ ]:
# TU CÓDIGO ACÁ

## 4 · Lo que descarté

## 5 · Dónde quedó